# 카페 메뉴 전처리

30개 브랜드 CSV 파일을 합치고, temp/is_coffee 빈 값 추론 후 브랜드+메뉴명 기준 중복 제거

In [1]:
import pandas as pd
import glob
import os
import re

CSV_FOLDER = './csv 파일'
OUTPUT_PATH = './output/cafe_menu_processed.csv'

# ─── 1. 파일 로드 및 합치기 ────────────────────────────────────────────────
files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*_menu.csv')))
print(f"발견된 파일: {len(files)}개")

dfs = []
for path in files:
    df = pd.read_csv(path, encoding='utf-8-sig')
    # 필요한 5개 컬럼만 선택 (없는 컬럼은 NaN으로 생성)
    for col in ['brand', 'name', 'temp', 'category', 'is_coffee']:
        if col not in df.columns:
            df[col] = None
    dfs.append(df[['brand', 'name', 'temp', 'category', 'is_coffee']])
    print(f"  {os.path.basename(path):<35} {len(df):3d}행")

combined = pd.concat(dfs, ignore_index=True)
print(f"\n합친 후 총 행 수: {len(combined)}행")

# ─── 2. name 비어있는 행 제거 ──────────────────────────────────────────────
combined = combined.dropna(subset=['name'])
combined = combined[combined['name'].str.strip() != '']
print(f"name 비어있는 행 제거 후: {len(combined)}행")

# ─── 3. 이름 정리 (괄호, 태그 등) ────────────────────────────────────────
def clean_name(name):
    name = str(name).strip()
    name = re.sub(r'<br\s*/?>', ' ', name, flags=re.IGNORECASE)
    name = re.sub(r'</br>', ' ', name)
    name = re.sub(r'NEW\)\s*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\[[^\]]{1,10}\]\s*', '', name)           # [인기], [NEW] 등
    name = re.sub(r'\s*\(HOT/ICE\)\s*', ' ', name)           # (HOT/ICE) 제거
    name = re.sub(r'\s*[\(（][A-Za-z\s]{1,10}[\)）]\s*', ' ', name)  # 영문 괄호
    name = re.sub(r'\s*[\(（][LMS][\)）]\s*', ' ', name)     # (L), (M), (S)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

combined['name'] = combined['name'].apply(clean_name)

# ─── 4. 브랜드+이름 기준 중복 제거 (같은 브랜드 내 동일 메뉴 제거) ──────
# 중복이 있을 때 데이터가 더 많은 행 우선 유지 (temp/category가 채워진 것)
combined['_null_count'] = combined[['temp', 'category', 'is_coffee']].isnull().sum(axis=1)
combined = combined.sort_values('_null_count').drop_duplicates(subset=['brand', 'name'], keep='first')
combined = combined.drop(columns=['_null_count'])
print(f"브랜드+이름 중복 제거 후: {len(combined)}행")

# ─── 5. temp 정규화 및 빈 값 추론 ────────────────────────────────────────
ICE_KW   = ['아이스', 'iced', 'ice', '냉', '스무디', '스무스', '아이디',
            '주스', '소다', '스퀴즈', '스쿼시', '스무디푸치노', '프라페']
HOT_KW   = ['따뜻', '핫초코', 'hot', '뜨거운']
COLD_BREW_KW = ['콜드브루', '냉브루']
ALWAYS_COLD_CAT = ['주스', '에이드', '스무디', '스무스', '스무디카테',
                   '프라페', '쉐이크', '요거트', '아이스크림', '빙수']

def normalize_temp(t):
    if pd.isna(t) or str(t).strip() == '':
        return None
    t = str(t).strip().upper().replace(' ', '')
    if t in ['ICE', '아이스', 'ICED']:
        return 'ICE'
    if t in ['HOT', '핫', '따뜻', '온']:
        return 'HOT'
    if t in ['HOT/ICE', 'ICE/HOT', 'BOTH', 'HOT&ICE', 'HOTICE', 'ICEHOT']:
        return 'BOTH'
    if 'COLD' in t or '콜드' in t:
        return 'COLD_BREW'
    return t  # 그 외는 그대로

def infer_temp(row):
    t = normalize_temp(row.get('temp'))
    if t in ['ICE', 'HOT', 'BOTH', 'COLD_BREW']:
        return t

    name = str(row['name']).lower()
    cat  = str(row.get('category', '') or '').lower()

    # 카테고리로 추론
    if any(k in cat for k in ALWAYS_COLD_CAT):
        return 'ICE'

    # 메뉴명으로 추론
    if any(k in name for k in ICE_KW):
        return 'ICE'
    if any(k in name for k in HOT_KW):
        return 'HOT'
    if any(k in name for k in COLD_BREW_KW):
        return 'COLD_BREW'

    return 'BOTH'  # 추론 불가 시 HOT/ICE 모두 가능

combined['temp'] = combined.apply(infer_temp, axis=1)
print(f"\ntemp 분포:\n{combined['temp'].value_counts().to_string()}")

# ─── 6. is_coffee 추론 ────────────────────────────────────────────────────
COFFEE_KW = ['아메리카노', '에스프레소', '카푸치노', '라떼', '마끼아또',
             '콜드브루', '카페인', '카페 라', '드리퍼',
             'americano', 'espresso', 'coffee', 'latte']

NOT_COFFEE_CAT_KW = ['논커피', '주스', '에이드', '스무디', '스무스', '요거트',
                     '케이크', '디저트', 'tea', '티', '베이커리',
                     'ade', 'juice', 'smoothie', 'shake', 'dessert']

NOT_COFFEE_NAME_KW = ['케이크', '주스', '에이드', '소다', '스무디', '요거트',
                      '스무스', '쉐이크', '스무디카테', '티', '허브티', '라즈베리티',
                      '샌드위치', '와플', '쿠키', '마카롱', '빙수',
                      '아이스크림', '치즈', '슬러시', '바나나', '딸기', '망고',
                      '레몬', '자몽', '피치', '복숭아', '키위', '포도',
                      '루이보스', '루이 보스', '루이보스티',
                      '링귀네', '링귀 네']

def fix_is_coffee(row):
    name = str(row['name']).lower()
    cat  = str(row.get('category', '') or '').lower()
    v    = row['is_coffee']

    # 1단계: 카테고리가 비커피 → False
    if any(k in cat for k in NOT_COFFEE_CAT_KW):
        return False

    # 2단계: 커피 핵심 키워드 → True
    if any(k in name for k in COFFEE_KW):
        return True

    # 3단계: 메뉴명에 비커피 키워드 → False
    if any(k in name for k in NOT_COFFEE_NAME_KW):
        return False

    # 기존 값 사용, 없으면 False
    if pd.isna(v):
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, str):
        return v.strip().upper() in ['TRUE', '1', 'YES']
    return bool(v)

combined['is_coffee'] = combined.apply(fix_is_coffee, axis=1)
print(f"\nis_coffee 분포:\n{combined['is_coffee'].value_counts().to_string()}")

# ─── 7. 최종 정리 및 저장 ─────────────────────────────────────────────────
combined = combined[['brand', 'name', 'temp', 'category', 'is_coffee']]
combined = combined.sort_values(['brand', 'category', 'name']).reset_index(drop=True)

os.makedirs('./output', exist_ok=True)
combined.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print("\n" + "=" * 45)
print("          최종 전처리 완료 리포트")
print("=" * 45)
print(f"총 메뉴 수       : {len(combined)}개")
print(f"브랜드 수        : {combined['brand'].nunique()}개")
print(f"\n[temp 분포]")
for k, v in combined['temp'].value_counts().items():
    print(f"  {k:<12}: {v}개")
print(f"\n[is_coffee 분포]")
for k, v in combined['is_coffee'].value_counts().items():
    label = '커피' if k else '비커피'
    print(f"  {label:<10}: {v}개")
print(f"\n[브랜드별 메뉴 수]")
print(combined['brand'].value_counts().to_string())
print(f"\n저장 완료 → {OUTPUT_PATH}")


발견된 파일: 30개
  amasvin_menu.csv                     81행
  baekuk_menu.csv                      75행
  banapresso_menu.csv                 187행
  blushaak_menu.csv                   115행
  bombom_menu.csv                     130행
  caffebene_menu.csv                   74행
  caffeine_menu.csv                    62행
  coffeebay_menu.csv                  102행
  compose_coffee_menu.csv             144행
  dalcu_menu.csv                       88행
  dessert39_menu.csv                  254행
  ediya_menu.csv                      315행
  gongcha_menu.csv                    130행
  hasamdong_menu.csv                  133행
  hiocoffee_menu.csv                  160행
  hollys_menu.csv                      70행
  mammoth_menu.csv                    115행
  mega_coffee_menu.csv                148행
  oozy_menu.csv                       346행
  paikdabang_menu.csv                 269행
  palgong_menu.csv                    208행
  pascucci_menu.csv                    83행
  paulbassett_menu.csv                109행


is_coffee 분포:
is_coffee
False    2133
True     1261

          최종 전처리 완료 리포트
총 메뉴 수       : 3394개
브랜드 수        : 30개

[temp 분포]
  ICE         : 1599개
  BOTH        : 1346개
  HOT         : 382개
  COLD_BREW   : 67개

[is_coffee 분포]
  비커피       : 2133개
  커피        : 1261개

[브랜드별 메뉴 수]
brand
요거프레소     258
빽다방       191
바나프레소     187
커피에반하다    181
우지커피      173
하이오커피     160
컴포즈커피     144
디저트39     134
하삼동커피     129
이디야       128
카페봄봄      122
블루샥       115
매머드커피     113
텐퍼센트커피    109
폴바셋       106
메가커피      105
커피베이       99
팔공티        88
더벤티        87
파스쿠찌       83
더리터        82
투썸플레이스     76
백억커피       75
달리는커피      74
아마스빈       71
할리스        70
탐앤탐스       67
카페인중독      60
카페베네       56
공차         51

저장 완료 → ./output/cafe_menu_processed.csv


# 전처리2(ㄹㅇ 최종버전)

In [10]:
import pandas as pd
import glob
import os
import re

CSV_FOLDER = './csv 파일'
OUTPUT_PATH = './output/cafe_menu_processed.csv'

In [11]:
# ─── 1. 파일 로드 및 합치기 ────────────────────────────────────────────────
files = sorted(glob.glob(os.path.join(CSV_FOLDER, '*_menu.csv')))
print(f"발견된 파일: {len(files)}개")

dfs = []
for path in files:
    df = pd.read_csv(path, encoding='utf-8-sig')
    # 필요한 5개 컬럼만 선택 (없는 컬럼은 NaN으로 생성)
    for col in ['brand', 'name', 'temp', 'category', 'is_coffee']:
        if col not in df.columns:
            df[col] = None
    dfs.append(df[['brand', 'name', 'temp', 'category', 'is_coffee']])
    print(f"  {os.path.basename(path):<35} {len(df):3d}행")

combined = pd.concat(dfs, ignore_index=True)
print(f"\n합친 후 총 행 수: {len(combined)}행")

발견된 파일: 30개
  amasvin_menu.csv                     81행
  baekuk_menu.csv                      75행
  banapresso_menu.csv                 187행
  blushaak_menu.csv                   115행
  bombom_menu.csv                     130행
  caffebene_menu.csv                   74행
  caffeine_menu.csv                    62행
  coffeebay_menu.csv                  102행
  compose_coffee_menu.csv             144행
  dalcu_menu.csv                       88행
  dessert39_menu.csv                  254행
  ediya_menu.csv                      315행
  gongcha_menu.csv                    130행
  hasamdong_menu.csv                  133행
  hiocoffee_menu.csv                  160행
  hollys_menu.csv                      70행
  mammoth_menu.csv                    115행
  mega_coffee_menu.csv                148행
  oozy_menu.csv                       346행
  paikdabang_menu.csv                 269행
  palgong_menu.csv                    208행
  pascucci_menu.csv                    83행
  paulbassett_menu.csv                109행

In [12]:
# ─── 2. name 비어있는 행 제거 ──────────────────────────────────────────────
combined = combined.dropna(subset=['name'])
combined = combined[combined['name'].str.strip() != '']
print(f"name 비어있는 행 제거 후: {len(combined)}행")

name 비어있는 행 제거 후: 4265행


In [13]:
# ─── 3. 이름 정리 (괄호, 태그 등) ────────────────────────────────────────
def clean_name(name):
    name = str(name).strip()
    name = re.sub(r'<br\s*/?>', ' ', name, flags=re.IGNORECASE)
    name = re.sub(r'</br>', ' ', name)
    name = re.sub(r'NEW\)\s*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\[[^\]]{1,10}\]\s*', '', name)           # [인기], [NEW] 등
    name = re.sub(r'\s*\(HOT/ICE\)\s*', ' ', name)           # (HOT/ICE) 제거
    name = re.sub(r'\s*[\(（][A-Za-z\s]{1,10}[\)）]\s*', ' ', name)  # 영문 괄호
    name = re.sub(r'\s*[\(（][LMS][\)）]\s*', ' ', name)     # (L), (M), (S)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

combined['name'] = combined['name'].apply(clean_name)

In [14]:
# ─── 4. temp 정규화 및 빈 값 추론 (중복 제거 전에 먼저 실행) ─────────────
# HOT 아메리카노와 ICE 아메리카노는 서로 다른 제품이므로 temp 먼저 확정 후 중복 제거
ICE_KW        = ['아이스', 'iced', 'ice', '냉', '스무디', '스무스', '에이드',
                 '주스', '소다', '스퀴즈', '스쿼시', '스무디푸치노', '프라페']
HOT_KW        = ['따뜻', '핫초코', 'hot', '뜨거운']
COLD_BREW_KW  = ['콜드브루', '냉브루']
ALWAYS_COLD_CAT = ['주스', '에이드', '스무디', '스무스', '스무디카테',
                   '프라페', '쉐이크', '요거트', '아이스크림', '빙수']

def normalize_temp(t):
    if pd.isna(t) or str(t).strip() == '':
        return None
    t = str(t).strip().upper().replace(' ', '')
    if t in ['ICE', '아이스', 'ICED']:
        return 'ICE'
    if t in ['HOT', '핫', '따뜻', '온']:
        return 'HOT'
    if t in ['HOT/ICE', 'ICE/HOT', 'BOTH', 'HOT&ICE', 'HOTICE', 'ICEHOT']:
        return 'BOTH'
    if 'COLD' in t or '콜드' in t:
        return 'COLD_BREW'
    return t

def infer_temp(row):
    t = normalize_temp(row.get('temp'))
    if t in ['ICE', 'HOT', 'BOTH', 'COLD_BREW']:
        return t
    name = str(row['name']).lower()
    cat  = str(row.get('category', '') or '').lower()
    if any(k in cat for k in ALWAYS_COLD_CAT):
        return 'ICE'
    if any(k in name for k in ICE_KW):
        return 'ICE'
    if any(k in name for k in HOT_KW):
        return 'HOT'
    if any(k in name for k in COLD_BREW_KW):
        return 'COLD_BREW'
    return 'BOTH'

combined['temp'] = combined.apply(infer_temp, axis=1)
print(f"\ntemp 정규화 후:\n{combined['temp'].value_counts().to_string()}")


temp 정규화 후:
temp
ICE          2050
BOTH         1545
HOT           583
COLD_BREW      87


In [15]:
# ─── 5. brand + name + temp 기준 중복 제거 ───────────────────────────────
# A카페 아메리카노 HOT / ICE → 둘 다 유지
# A카페 아메리카노 ICE / ICE (진짜 중복) → 하나만 유지
combined['_null_count'] = combined[['category', 'is_coffee']].isnull().sum(axis=1)
combined = combined.sort_values('_null_count').drop_duplicates(
    subset=['brand', 'name', 'temp'], keep='first'
)
combined = combined.drop(columns=['_null_count'])
print(f"brand+name+temp 중복 제거 후: {len(combined)}행")

brand+name+temp 중복 제거 후: 3590행


In [16]:
# ─── 6. is_coffee 추론 ────────────────────────────────────────────────────
COFFEE_KW = ['아메리카노', '에스프레소', '카푸치노', '라떼', '마끼아또',
             '콜드브루', '카페인', '카페 라', '드리퍼',
             'americano', 'espresso', 'coffee', 'latte']

NOT_COFFEE_CAT_KW = ['논커피', '주스', '에이드', '스무디', '스무스', '요거트',
                     '케이크', '디저트', 'tea', '티', '베이커리',
                     'ade', 'juice', 'smoothie', 'shake', 'dessert']

NOT_COFFEE_NAME_KW = ['케이크', '주스', '에이드', '소다', '스무디', '요거트',
                      '스무스', '쉐이크', '스무디카테', '티', '허브티', '라즈베리티',
                      '샌드위치', '와플', '쿠키', '마카롱', '빙수',
                      '아이스크림', '치즈', '슬러시', '바나나', '딸기', '망고',
                      '레몬', '자몽', '피치', '복숭아', '키위', '포도',
                      '루이보스', '루이 보스', '루이보스티',
                      '링귀네', '링귀 네']

def fix_is_coffee(row):
    name = str(row['name']).lower()
    cat  = str(row.get('category', '') or '').lower()
    v    = row['is_coffee']

    # 1단계: 카테고리가 비커피 → False
    if any(k in cat for k in NOT_COFFEE_CAT_KW):
        return False

    # 2단계: 커피 핵심 키워드 → True
    if any(k in name for k in COFFEE_KW):
        return True

    # 3단계: 메뉴명에 비커피 키워드 → False
    if any(k in name for k in NOT_COFFEE_NAME_KW):
        return False

    # 기존 값 사용, 없으면 False
    if pd.isna(v):
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, str):
        return v.strip().upper() in ['TRUE', '1', 'YES']
    return bool(v)

combined['is_coffee'] = combined.apply(fix_is_coffee, axis=1)
print(f"\nis_coffee 분포:\n{combined['is_coffee'].value_counts().to_string()}")


is_coffee 분포:
is_coffee
False    2212
True     1378


In [17]:
# ─── 7. 최종 정리 및 저장 ─────────────────────────────────────────────────
combined = combined[['brand', 'name', 'temp', 'category', 'is_coffee']]
combined = combined.sort_values(['brand', 'category', 'name']).reset_index(drop=True)

os.makedirs('./output', exist_ok=True)
combined.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print("\n" + "=" * 45)
print("          최종 전처리 완료 리포트")
print("=" * 45)
print(f"총 메뉴 수       : {len(combined)}개")
print(f"브랜드 수        : {combined['brand'].nunique()}개")
print(f"\n[temp 분포]")
for k, v in combined['temp'].value_counts().items():
    print(f"  {k:<12}: {v}개")
print(f"\n[is_coffee 분포]")
for k, v in combined['is_coffee'].value_counts().items():
    label = '커피' if k else '비커피'
    print(f"  {label:<10}: {v}개")
print(f"\n[브랜드별 메뉴 수]")
print(combined['brand'].value_counts().to_string())
print(f"\n저장 완료 → {OUTPUT_PATH}")


          최종 전처리 완료 리포트
총 메뉴 수       : 3590개
브랜드 수        : 30개

[temp 분포]
  ICE         : 1767개
  BOTH        : 1297개
  HOT         : 459개
  COLD_BREW   : 67개

[is_coffee 분포]
  비커피       : 2212개
  커피        : 1378개

[브랜드별 메뉴 수]
brand
빽다방       260
요거프레소     258
바나프레소     187
커피에반하다    181
우지커피      173
하이오커피     160
메가커피      148
컴포즈커피     144
디저트39     136
하삼동커피     129
이디야       128
카페봄봄      122
블루샥       115
매머드커피     113
팔공티       113
텐퍼센트커피    109
폴바셋       106
커피베이       99
달리는커피      88
더벤티        87
파스쿠찌       83
더리터        82
공차         76
투썸플레이스     76
백억커피       75
카페베네       74
아마스빈       71
할리스        70
탐앤탐스       67
카페인중독      60

저장 완료 → ./output/cafe_menu_processed.csv
